In [13]:
import pandas as pd
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from datetime import datetime
from dateutil.relativedelta import relativedelta

def one_period_returns(P: np.ndarray) -> np.ndarray:
    return P[:-1] / P[1:] - 1


def momentum_stats(P: np.ndarray, S: int, T: int) -> dict:

    if len(P) < S + T + 1:
        return {
            "R_cum": np.nan,
            "R_mean": np.nan,
            "sigma": np.nan,
            "R_riskadj": np.nan
        }

    r = one_period_returns(P)

    window = r[S:S + T]

    R_cum = P[S] / P[S + T] - 1

    R_mean = window.mean()

    sigma2 = ((window - R_mean) ** 2).sum() / (T - 1)

    sigma = np.sqrt(sigma2)

    R_riskadj = R_mean / sigma if sigma > 0 else np.nan

    return {
        "R_cum": R_cum,
        "R_mean": R_mean,
        "sigma": sigma,
        "R_riskadj": R_riskadj
    }


def download_prices(
    tickers: list[str],
    end_date: str,
    years: int = 3
) -> pd.DataFrame:

    end = pd.Timestamp(end_date)

    start = end - pd.DateOffset(years=years)

    print("=" * 70)
    print("DOWNLOAD SETTINGS")
    print("=" * 70)
    print(f"Requested end date   : {end.date()}")
    print(f"Requested start date : {start.date()}")
    print(f"Sample length        : {years} years")
    print("=" * 70)

    data = yf.download(
        tickers,
        start=start.strftime("%Y-%m-%d"),
        end=(end + pd.Timedelta(days=1)).strftime("%Y-%m-%d"),
        auto_adjust=True,
        progress=False,
        group_by="column",
        threads=True
    )

    if isinstance(data.columns, pd.MultiIndex):

        if "Close" in data.columns.get_level_values(0):
            data = data["Close"]

        elif "Close" in data.columns.get_level_values(1):
            data = data.xs("Close", axis=1, level=1)

        else:
            raise ValueError(
                "Could not find Close prices in yfinance response."
            )

    else:

        if "Close" in data.columns:
            data = data[["Close"]]
            data.columns = [tickers[0]]

    data = data.sort_index()
    data = data.dropna(how="all")

    if not data.empty:
        print(f"Actual first date   : {data.index[0].date()}")
        print(f"Actual last date    : {data.index[-1].date()}")
        print(f"Trading days        : {len(data)}")
        print(f"Tickers requested   : {len(tickers)}")
        print(f"Columns downloaded  : {len(data.columns)}")

    return data


def resample_monthly(prices: pd.DataFrame) -> pd.DataFrame:

    monthly = prices.resample("ME").last()

    return monthly.dropna(how="all")

def rank_universe(
    prices: pd.DataFrame,
    S: int = 1,
    T: int = 6,
    min_history: int | None = None,
) -> pd.DataFrame:
    if min_history is None:
        min_history = S + T + 1
 
    table = []
    for ticker in prices.columns:
        series = prices[ticker].dropna()
        if len(series) < min_history:
            continue
 
        P = series.to_numpy()[::-1]
        stats = momentum_stats(P, S, T)
        stats["ticker"] = ticker
        table.append(stats)
 
    result = pd.DataFrame(table).set_index("ticker")
    result = result.sort_values("R_riskadj", ascending=False)
    result["rank"] = range(1, len(result) + 1)
 
    return result[["rank", "R_cum", "R_mean", "sigma", "R_riskadj"]]

def get_eps_history(ticker: str, quarters_needed: int = 12) -> pd.Series:
    """
    Pull historical quarterly EPS (most recent first) via yfinance's
    earnings_dates, which typically exposes more history than
    quarterly_income_stmt.
    """
    tk = yf.Ticker(ticker)
    try:
        df = tk.get_earnings_dates(limit=quarters_needed + 8)  # pad for future/blank rows
    except Exception:
        return pd.Series(dtype=float)

    if df is None or df.empty or "Reported EPS" not in df.columns:
        return pd.Series(dtype=float)

    eps = df["Reported EPS"].dropna()
    return eps.sort_index(ascending=False)  # most recent first


def sue_stat(eps: pd.Series) -> float:
    """
    SUE_i = (E_i - E_i') / sigma_i
    """
    if len(eps) < 12:
        return np.nan

    vals = eps.to_numpy()

    E_i = vals[0]
    E_i_prime = vals[4]

    surprises = vals[:8] - vals[4:12]
    sigma_i = surprises.std(ddof=1)

    return (E_i - E_i_prime) / sigma_i if sigma_i > 0 else np.nan

def add_sue_signal(top50: pd.DataFrame) -> pd.DataFrame:
    """
    Fetch EPS history and compute SUE for each ticker in top50.
    Adds a 'SUE' column.
    """
    top50 = top50.copy()
    sue_values = {}
    for ticker in top50.index:
        eps = get_eps_history(ticker)
        sue_values[ticker] = sue_stat(eps)

    top50["SUE"] = pd.Series(sue_values)
    return top50


def combine_ranks(top50: pd.DataFrame) -> pd.DataFrame:

    result = top50.copy()

    # rank 1 = best (highest) on each signal; NaNs sink to the bottom
    result["momentum_rank_50"] = result["R_riskadj"].rank(
        ascending=False, method="min", na_option="bottom"
    )
    result["sue_rank_50"] = result["SUE"].rank(
        ascending=False, method="min", na_option="bottom"
    )

    result["combined_score"] = (
        result["momentum_rank_50"] + result["sue_rank_50"]
    ) / 2

    result = result.sort_values("combined_score", ascending=True)
    result["final_rank"] = range(1, len(result) + 1)

    return result

def top_n_actual_returns(
    monthly_prices: pd.DataFrame,
    ranking: pd.DataFrame,
    n: int = 10,
    rank_col: str = "rank",
) -> pd.DataFrame:

    top_tickers = ranking.nsmallest(
        n,
        rank_col
    ).index

    price_now = monthly_prices.iloc[-1]

    price_prev = monthly_prices.iloc[-2]

    rows = []

    for ticker in top_tickers:

        if pd.isna(price_prev.get(ticker)) or pd.isna(
            price_now.get(ticker)
        ):
            continue

        actual_return = (
            price_now[ticker]
            / price_prev[ticker]
            - 1
        )

        rows.append({
            "ticker": ticker,
            rank_col: ranking.loc[ticker, rank_col],
            "R_riskadj": ranking.loc[ticker, "R_riskadj"],
            "price_last_month": price_prev[ticker],
            "price_now": price_now[ticker],
            "actual_return": actual_return,
        })

    result = pd.DataFrame(rows).set_index("ticker")

    return result.sort_values(rank_col)

if __name__ == "__main__":

    END_DATE = "2026-07-20"

    SAMPLE_YEARS = 1

    S = 1
    T = 6

    TOP_N = 10

    universe = [
        "AAPL", "MSFT", "GOOGL", "AMZN", "META", "NVDA", "JPM", "XOM", "PG", "KO", "GOOG", "AVGO", "TSLA", "MU", "LLY", "AMD", "WMT",
        "V", "INTC", "JNJ", "MA", "ABBV", "ORCL", "PLTR", "CSCO", "COST", "CVX", "BAC", "LRCX", "AMAT", "MRK", "CAT", "DELL",
        "UNH", "GE", "MS", "PANW", "HD", "NFLX", "PM", "GS", "SNDK", "ANET", "RTX", "CRWD", "GEV", "WFC", "TXN", "KLAC", "TMO",
        "MRVL", "C", "AMGN", "IBM", "LIN", "QCOM", "STX", "AXP", "APH", "VZ", "CRM", "DE", "GILD", "ADI", "ABT", "DIS", "PEP",
        "MCD", "TMUS", "SCHW", "BLK", "T", "ETN", "WDC", "WELL", "NEE", "UNP", "PFE", "BA", "DHR", "BX", "COP", "TJX", "UBER", "ISRG",
        "NOW", "GLW", "NEM", "PLD", "VRTX", "CB", "FTNT", "BMY", "BKNG", "COF", "PH", "LMT", "PGR", "SPGI", "MDT", "MO", "ACN", "HOOD",
        "CVS", "LOW", "APP", "MPC", "VLO", "SBUX", "ADP", "FCX", "SYK", "EQIX", "MCK", "PSX", "BNY", "SO", "VRT", "PWR", "TT", "CME",
        "ABNB", "HCA", "ADBE", "CEG", "GD", "USB", "PNC", "MAR", "DUK", "HWM", "DDOG", "JCI", "KKR", "MMM", "ELV", "WMB", "MNST", "CSX",
        "EMR", "ICE", "LITE", "CDNS", "DASH", "WM", "AMT", "UPS", "BE", "MCO", "MRSH", "HPE", "REGN", "CTAS", "CMCSA", "SHW", "SNPS",
        "SPG", "INTU", "MDLZ", "ECL", "WBD", "SLB", "ITW", "MSI", "TRV", "ROST", "APO", "GM", "EOG", "MRNA", "CVNA", "NOC", "CMI", "CI",
        "TGT", "NSC", "DLR", "FDX", "KMI", "ORLY", "HLT", "CL", "MPWR", "HON", "RSG", "AEP", "BSX", "URI", "APD", "RCL", "TER", "TEL",
        "TDG", "MET", "COHR", "AON", "TRGP", "COR", "PCAR", "NXPI", "GWW",  "KEYS", "AJG", "TFC", "CRH", "ALL", "FAST", "AFL", "FIX", "BKR",
        "OKE", "OXY", "AME", "PSA", "NUE", "D", "GRMN", "DAL", "DVN",  "CTVA", "NKE", "O", "COIN", "HONA", "SRE", "NDAQ", "CIEN", "F",
        "FANG", "CAH", "EW", "BDX", "STT", "WAB", "EBAY", "ETR", "VMRK",  "FITB", "ROK", "A", "VST", "XYZ", "AZO", "ADSK", "CARR", "WDAY",
        "AMP", "PYPL", "XEL", "VTR", "LHX", "HUM", "IQV", "EXC", "KDP",  "FERG", "FLEX", "VEEV", "WAT", "IBKR", "CMG", "ARES", "MCHP",
        "IDXX", "CBRE", "PAYX", "PRU", "DHI", "MSCI", "ADM", "LYV", "AIG", "SYY", "TTWO", "ED", "YUM", "NTAP", "UAL", "ILMN", "AXON", "P",
        "ODFL", "EL", "ROP", "TKO", "MLM", "IRM", "HIG", "KR", "PEG",  "KVUE", "HSY", "DXCM", "EXPE", "STLD", "WEC", "EME", "BIIB", "KMB",
        "JBL", "ACGL", "MTB", "VMC", "EQT", "RMD", "NTRS", "HBAN", "CNC",  "CCI", "RJF", "CCL", "ZTS", "RDDT", "GEHC", "MTD", "EXR", "HPQ",
        "IR", "ON", "KHC", "TDY", "PCG", "AEE", "WTW", "CBOE", "ECHO", "CFG", "WSM", "HAL", "SMCI", "FOXA", "DG", "VRSN", "CPRT", "AWK",
        "ATO", "WRB", "CTSH", "VICI", "Q", "WST", "DTE", "OTIS", "DGX", "FE", "CPAY", "INCY", "LVS", "FFIV", "FOX", "CINF", "LH", "XYL",
        "DOV", "ES", "CNP", "PPL", "FISV", "TPL", "EXPD", "SW", "PFG", "DRI", "HUBB", "PPG", "RF", "ULTA", "SYF", "VLTO", "PHM", "CHD",
        "TPR", "GPN", "VRSK", "JBHT", "TROW", "IFF", "KEY", "L", "CASY", "NRG", "DLTR", "FSLR", "EIX", "BG", "PKG", "BRO", "OMC", "RL",
        "LUV", "DOW", "STE", "CMS", "EXE", "STZ", "LEN", "AMCR", "BBY", "FICO", "VTRS", "SNA", "CHTR", "LYB", "NI", "GIS", "IP", "EFX",
        "SBAC", "BR", "CDW", "EVRG", "ESS", "CF", "TSN", "FIS", "ZBH", "GPC", "NDSN", "DD", "FDXF", "CHRW", "TSCO", "ZBRA", "AKAM", "FTV",
        "NVR", "BEN", "LNT", "J", "IEX", "NWS", "GEN", "INVH", "RVTY", "BALL", "ROL", "NWSA", "WY", "HST", "LDOS", "PTC", "KIM", "SOLV",
        "APA", "DOC", "EG", "MAA", "SWK", "TRMB", "ALB", "MAS", "REG", "IVZ", "CRL", "TYL", "SWKS", "TXT", "AIZ", "MKC", "ALLE", "AVY",
        "GL", "SJM", "LII", "GDDY", "HAS", "ERIE", "BAX", "GNRC", "DVA", "UDR", "CSGP", "LULU", "BXP", "TECH", "PSKY", "PNW", "IT",
        "HRL", "ALGN", "DECK", "HII", "AES", "UHS", "CLX", "COO", "JKHY", "CPT", "FDS", "DPZ", "MGM", "HSIC", "PODD", "FRT", "ARE", "APTV",
        "PNR", "WYNN", "AOS", "MOS", "NCLH"
    ]

    daily_prices = download_prices(
        universe,
        end_date=END_DATE,
        years=SAMPLE_YEARS
    )

    monthly_prices = resample_monthly(
        daily_prices
    )

    ranking = rank_universe(
        monthly_prices,
        S=S,
        T=T
    )

    top50 = ranking.head(50).copy()

    top50 = add_sue_signal(top50)

    combined = combine_ranks(top50)

    top20_combined = top_n_actual_returns(
        monthly_prices,
        combined,
        n=10,
        rank_col="final_rank",
    )

    print("\n" + "=" * 70)
    print("TOP 20 (MOMENTUM + SUE COMBINED) — ACTUAL RETURNS")
    print("=" * 70)

    print(
        top20_combined.to_string(
            float_format=lambda x: f"{x:.4f}"
        )
    )

    combined_avg = top20_combined["actual_return"].mean()

    print(
        f"\nAverage return across top 20 combined "
        f"(equal-weighted): "
        f"{combined_avg * 100:+.2f}%"
    )

    print("\n" + "=" * 70)
    print("TOP 50 RE-RANKED BY MOMENTUM + SUE COMBINED SCORE")
    print("=" * 70)

    cols = ["final_rank", "R_riskadj", "momentum_rank_50", "SUE", "sue_rank_50", "combined_score"]
    print(combined[cols].head(20).to_string(float_format=lambda x: f"{x:.4f}"))

    print("\n" + "=" * 70)
    print("TOP 10 MOMENTUM RANKS")
    print("=" * 70)

    print(
        ranking.head(10).to_string(
            float_format=lambda x: f"{x:.4f}"
        )
    )

    top10 = top_n_actual_returns(
        monthly_prices,
        ranking,
        n=TOP_N
    )

    print("\n" + "=" * 70)
    print("TOP 10 MOMENTUM STOCKS")
    print("=" * 70)

    print(
        top10.to_string(
            float_format=lambda x: f"{x:.4f}"
        )
    )

    whole = top10["actual_return"].mean()

    print(
        f"\nAverage return across top {TOP_N} "
        f"(equal-weighted): "
        f"{whole * 100:+.2f}%"
    )

DOWNLOAD SETTINGS
Requested end date   : 2026-07-20
Requested start date : 2025-07-20
Sample length        : 1 years


$BLK: possibly delisted; no price data found  (1d 2025-07-20 -> 2026-07-21)
$AMZN: possibly delisted; no price data found  (1d 2025-07-20 -> 2026-07-21)
$CLX: possibly delisted; no price data found  (1d 2025-07-20 -> 2026-07-21)
$AMCR: possibly delisted; no price data found  (1d 2025-07-20 -> 2026-07-21)

4 Failed downloads:
['BLK', 'AMZN', 'CLX', 'AMCR']: possibly delisted; no price data found  (1d 2025-07-20 -> 2026-07-21)


Actual first date   : 2025-07-21
Actual last date    : 2026-07-20
Trading days        : 251
Tickers requested   : 501
Columns downloaded  : 501

TOP 20 (MOMENTUM + SUE COMBINED) — ACTUAL RETURNS
        final_rank  R_riskadj  price_last_month  price_now  actual_return
ticker                                                                   
GLW              1     1.0347          254.9499   152.8123        -0.4006
TRGP             2     1.0599          266.8909   281.0347         0.0530
COHR             3     1.0154          394.4700   285.4000        -0.2765
FFIV             4     1.3209          415.9600   411.9200        -0.0097
TGT              5     1.4032          129.6151   138.5267         0.0688
LRCX             6     0.9711          433.3300   306.7600        -0.2921
TER              7     1.1620          483.6542   333.6319        -0.3102
STX              8     0.8757          965.0000   802.4500        -0.1684
EBAY             9     0.8683          111.4113   113.8439       

In [44]:
import numpy as np
import pandas as pd
import yfinance as yf
from universe import universe

start = "2025-12-20"
end = "2026-09-20"

result = pd.DataFrame(columns=['cumulative', 'mean_return', 'std_dev', 'risk_adjusted'])
failed = []

for star in universe:
    try:
        data = yf.download(star, start=start, end=end, auto_adjust=True,
                           progress=False, interval="1mo", multi_level_index=False)
    except Exception as e:
        failed.append((star, str(e)))
        continue

    if data.empty or 'Close' not in data:
        failed.append((star, "no data"))
        continue

    prices = data['Close'].dropna().iloc[:-1]
    if len(prices) < 3:                      # need enough points for pct_change + std
        failed.append((star, f"only {len(prices)} rows"))
        continue

    monthly_ret = prices.pct_change().dropna()
    mean_ret = monthly_ret.mean()
    std_ret = monthly_ret.std()

    result.loc[star] = [
        prices.iloc[-1] / prices.iloc[0] - 1,
        mean_ret,
        std_ret,
        mean_ret / std_ret if std_ret > 0 else np.nan,
    ]


result = result.astype(float)
print(result.sort_values('risk_adjusted', ascending=False).head(55))

print(f"\n{len(failed)} tickers skipped:")
for t, reason in failed:
    print(f"  {t}: {reason}")

KeyboardInterrupt: 

In [23]:
print(results)

{'AAPL': np.float64(4.0359349484336065), 'MSFT': np.float64(-0.4702675854113802), 'GOOGL': np.float64(8.782393073135284), 'AMZN': np.float64(3.5683326721191406), 'META': np.float64(-0.23817502597774856), 'NVDA': np.float64(3.761075539553319), 'JPM': np.float64(6.7406278865364575), 'XOM': np.float64(1.3797752992411059), 'PG': np.float64(-0.47359792397376427), 'KO': np.float64(0.8573144415483206), 'GOOG': np.float64(8.843101468199306), 'AVGO': np.float64(10.732979042464706), 'TSLA': np.float64(5.940837224324544), 'MU': np.float64(31.209751701037778), 'LLY': np.float64(14.844074161552692), 'AMD': np.float64(16.007495880126953), 'WMT': np.float64(2.037308975496561), 'V': np.float64(4.168602155297376), 'INTC': np.float64(2.9200005531311035), 'JNJ': np.float64(4.250538370769852), 'MA': np.float64(4.832998832797062), 'ABBV': np.float64(2.5477833502021014), 'ORCL': np.float64(-0.683081089644889), 'PLTR': np.float64(3.3133331139882407), 'CSCO': np.float64(2.9294653526782732), 'COST': np.float64

In [25]:
import pandas as pd
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from datetime import datetime
from dateutil.relativedelta import relativedelta

start = "2026-02-20"
end = "2026-08-20"

p1 = 100
p0 = 105
pit = p0/p1 - 1

universe = [
        "AAPL", "MSFT", "GOOGL", "AMZN", "META", "NVDA", "JPM", "XOM", "PG", "KO", "GOOG", "AVGO", "TSLA", "MU", "LLY", "AMD", "WMT",
        "V", "INTC", "JNJ", "MA", "ABBV", "ORCL", "PLTR", "CSCO", "COST", "CVX", "BAC", "LRCX", "AMAT", "MRK", "CAT", "DELL",
        "UNH", "GE", "MS", "PANW", "HD", "NFLX", "PM", "GS", "SNDK", "ANET", "RTX", "CRWD", "GEV", "WFC", "TXN", "KLAC", "TMO",
        "MRVL", "C", "AMGN", "IBM", "LIN", "QCOM", "STX", "AXP", "APH", "VZ", "CRM", "DE", "GILD", "ADI", "ABT", "DIS", "PEP",
        "MCD", "TMUS", "SCHW", "BLK", "T", "ETN", "WDC", "WELL", "NEE", "UNP", "PFE", "BA", "DHR", "BX", "COP", "TJX", "UBER", "ISRG",
        "NOW", "GLW", "NEM", "PLD", "VRTX", "CB", "FTNT", "BMY", "BKNG", "COF", "PH", "LMT", "PGR", "SPGI", "MDT", "MO", "ACN", "HOOD",
        "CVS", "LOW", "APP", "MPC", "VLO", "SBUX", "ADP", "FCX", "SYK", "EQIX", "MCK", "PSX", "BNY", "SO", "VRT", "PWR", "TT", "CME",
        "ABNB", "HCA", "ADBE", "CEG", "GD", "USB", "PNC", "MAR", "DUK", "HWM", "DDOG", "JCI", "KKR", "MMM", "ELV", "WMB", "MNST", "CSX",
        "EMR", "ICE", "LITE", "CDNS", "DASH", "WM", "AMT", "UPS", "BE", "MCO", "MRSH", "HPE", "REGN", "CTAS", "CMCSA", "SHW", "SNPS",
        "SPG", "INTU", "MDLZ", "ECL", "WBD", "SLB", "ITW", "MSI", "TRV", "ROST", "APO", "GM", "EOG", "MRNA", "CVNA", "NOC", "CMI", "CI",
        "TGT", "NSC", "DLR", "FDX", "KMI", "ORLY", "HLT", "CL", "MPWR", "HON", "RSG", "AEP", "BSX", "URI", "APD", "RCL", "TER", "TEL",
        "TDG", "MET", "COHR", "AON", "TRGP", "COR", "PCAR", "NXPI", "GWW",  "KEYS", "AJG", "TFC", "CRH", "ALL", "FAST", "AFL", "FIX", "BKR",
        "OKE", "OXY", "AME", "PSA", "NUE", "D", "GRMN", "DAL", "DVN",  "CTVA", "NKE", "O", "COIN", "HONA", "SRE", "NDAQ", "CIEN", "F",
        "FANG", "CAH", "EW", "BDX", "STT", "WAB", "EBAY", "ETR", "VMRK",  "FITB", "ROK", "A", "VST", "XYZ", "AZO", "ADSK", "CARR", "WDAY",
        "AMP", "PYPL", "XEL", "VTR", "LHX", "HUM", "IQV", "EXC", "KDP",  "FERG", "FLEX", "VEEV", "WAT", "IBKR", "CMG", "ARES", "MCHP",
        "IDXX", "CBRE", "PAYX", "PRU", "DHI", "MSCI", "ADM", "LYV", "AIG", "SYY", "TTWO", "ED", "YUM", "NTAP", "UAL", "ILMN", "AXON", "P",
        "ODFL", "EL", "ROP", "TKO", "MLM", "IRM", "HIG", "KR", "PEG",  "KVUE", "HSY", "DXCM", "EXPE", "STLD", "WEC", "EME", "BIIB", "KMB",
        "JBL", "ACGL", "MTB", "VMC", "EQT", "RMD", "NTRS", "HBAN", "CNC",  "CCI", "RJF", "CCL", "ZTS", "RDDT", "GEHC", "MTD", "EXR", "HPQ",
        "IR", "ON", "KHC", "TDY", "PCG", "AEE", "WTW", "CBOE", "ECHO", "CFG", "WSM", "HAL", "SMCI", "FOXA", "DG", "VRSN", "CPRT", "AWK",
        "ATO", "WRB", "CTSH", "VICI", "Q", "WST", "DTE", "OTIS", "DGX", "FE", "CPAY", "INCY", "LVS", "FFIV", "FOX", "CINF", "LH", "XYL",
        "DOV", "ES", "CNP", "PPL", "FISV", "TPL", "EXPD", "SW", "PFG", "DRI", "HUBB", "PPG", "RF", "ULTA", "SYF", "VLTO", "PHM", "CHD",
        "TPR", "GPN", "VRSK", "JBHT", "TROW", "IFF", "KEY", "L", "CASY", "NRG", "DLTR", "FSLR", "EIX", "BG", "PKG", "BRO", "OMC", "RL",
        "LUV", "DOW", "STE", "CMS", "EXE", "STZ", "LEN", "AMCR", "BBY", "FICO", "VTRS", "SNA", "CHTR", "LYB", "NI", "GIS", "IP", "EFX",
        "SBAC", "BR", "CDW", "EVRG", "ESS", "CF", "TSN", "FIS", "ZBH", "GPC", "NDSN", "DD", "FDXF", "CHRW", "TSCO", "ZBRA", "AKAM", "FTV",
        "NVR", "BEN", "LNT", "J", "IEX", "NWS", "GEN", "INVH", "RVTY", "BALL", "ROL", "NWSA", "WY", "HST", "LDOS", "PTC", "KIM", "SOLV",
        "APA", "DOC", "EG", "MAA", "SWK", "TRMB", "ALB", "MAS", "REG", "IVZ", "CRL", "TYL", "SWKS", "TXT", "AIZ", "MKC", "ALLE", "AVY",
        "GL", "SJM", "LII", "GDDY", "HAS", "ERIE", "BAX", "GNRC", "DVA", "UDR", "CSGP", "LULU", "BXP", "TECH", "PSKY", "PNW", "IT",
        "HRL", "ALGN", "DECK", "HII", "AES", "UHS", "CLX", "COO", "JKHY", "CPT", "FDS", "DPZ", "MGM", "HSIC", "PODD", "FRT", "ARE", "APTV",
        "PNR", "WYNN", "AOS", "MOS", "NCLH"
    ]

results = {}

for star in universe:
    data = yf.download(
        star,
        start=start,
        end=end,
        auto_adjust=True,
        progress=False,
        interval="1mo",
        multi_level_index=False,
    )

    results[star] = ((data['Close'] - data['Open']) / data['Open']).mean() * 100

top10 = pd.Series(results).sort_values(ascending=False).head(10)
print(top10)




MRNA    28.270291
SNDK    25.922270
DELL    25.284792
MRVL    24.646053
MU      22.400841
INTC    21.418739
AMD     20.977000
STX     17.845999
PANW    17.218284
HPE     16.741287
dtype: float64


In [49]:
import numpy as np
import pandas as pd
import yfinance as yf
from pathlib import Path
from universe import universe

start = "2026-03-20"
end = "2026-09-20"
cache = Path(f"prices_{start}_{end}.pkl")


def get_prices(tickers, chunk=100):
    if cache.exists():
        return pd.read_pickle(cache)

    frames = []
    for i in range(0, len(tickers), chunk):
        batch = tickers[i:i + chunk]
        df = yf.download(batch, start=start, end=end, auto_adjust=True,
                         progress=False, interval="1mo", threads=True)["Close"]
        frames.append(df)

    prices = pd.concat(frames, axis=1)
    prices.to_pickle(cache)
    return prices


prices = get_prices(list(universe))

# Same logic as your loop, but vectorized across all tickers
prices = prices.iloc[:-1]                          # drop partial current month
prices = prices.dropna(axis=1, how="all")          # drop tickers with no data
prices = prices.loc[:, prices.count() >= 3]        # need enough rows

monthly_ret = prices.pct_change(fill_method=None)
mean_ret = monthly_ret.mean()
std_ret = monthly_ret.std()

result = pd.DataFrame({
    "cumulative": prices.ffill().iloc[-1] / prices.bfill().iloc[0] - 1,
    "mean_return": mean_ret,
    "std_dev": std_ret,
    "risk_adjusted": (mean_ret / std_ret).where(std_ret > 0),
})

print(result.sort_values("risk_adjusted", ascending=True).head(55))

        cumulative  mean_return   std_dev  risk_adjusted
Ticker                                                  
FDXF     -0.166053    -0.058694  0.010513      -5.583052
ROL      -0.347400    -0.100452  0.042635      -2.356114
LHX      -0.166031    -0.044153  0.023740      -1.859835
VICI     -0.106815    -0.027757  0.015142      -1.833110
TXT      -0.160305    -0.042385  0.030240      -1.401613
HONA     -0.284829    -0.150043  0.120462      -1.245560
EME      -0.175794    -0.046625  0.037938      -1.228973
MCD      -0.096266    -0.024825  0.020540      -1.208637
WMT      -0.201685    -0.053833  0.047689      -1.128836
GOOGL    -0.117586    -0.030502  0.027232      -1.120103
TSN      -0.132125    -0.034433  0.031343      -1.098573
MLM      -0.160933    -0.042302  0.039546      -1.069682
CRH      -0.197668    -0.052523  0.051272      -1.024382
PWR      -0.165694    -0.043424  0.046789      -0.928082
GOOG     -0.121297    -0.031346  0.034687      -0.903696
ALGN     -0.092154    -0.023601

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
from pathlib import Path
from universe import universe

start = "2026-03-20"
end = "2026-09-20"
cache = Path(f"prices_{start}_{end}.pkl")


def get_prices(tickers, chunk=100):
    if cache.exists():
        return pd.read_pickle(cache)

    frames = []
    for i in range(0, len(tickers), chunk):
        batch = tickers[i:i + chunk]
        df = yf.download(batch, start=start, end=end, auto_adjust=True,
                         progress=False, interval="1mo", threads=True)["Close"]
        frames.append(df)

    prices = pd.concat(frames, axis=1)
    prices.to_pickle(cache)
    return prices


prices = get_prices(list(universe))

# Same logic as your loop, but vectorized across all tickers
prices = prices.iloc[:-1]                          # drop partial current month
prices = prices.dropna(axis=1, how="all")          # drop tickers with no data
prices = prices.loc[:, prices.count() >= 3]        # need enough rows

monthly_ret = prices.pct_change(fill_method=None)
mean_ret = monthly_ret.mean()
std_ret = monthly_ret.std()

result = pd.DataFrame({
    "cumulative": prices.ffill().iloc[-1] / prices.bfill().iloc[0] - 1,
    "mean_return": mean_ret,
    "std_dev": std_ret,
    "risk_adjusted": (mean_ret / std_ret).where(std_ret > 0),
})

print(result.sort_values("risk_adjusted", ascending=True).head(55))



In [91]:
import numpy as np
import pandas as pd
import yfinance as yf
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from universe import universe

start = "2026-03-20"
end = "2026-09-20"
cache = Path(f"prices_{start}_{end}.pkl")


def get_prices(tickers, chunk=100):
    if cache.exists():
        return pd.read_pickle(cache)

    frames = []
    for i in range(0, len(tickers), chunk):
        batch = tickers[i:i + chunk]
        df = yf.download(batch, start=start, end=end, auto_adjust=True,
                         progress=False, interval="1mo", threads=True)["Close"]
        frames.append(df)

    prices = pd.concat(frames, axis=1)
    prices.to_pickle(cache)
    return prices


prices = get_prices(list(universe))

# Same logic as your loop, but vectorized across all tickers
prices = prices.iloc[:-1]                          # drop partial current month
prices = prices.dropna(axis=1, how="all")          # drop tickers with no data
prices = prices.loc[:, prices.count() >= 3]        # need enough rows

monthly_ret = prices.pct_change(fill_method=None)
mean_ret = monthly_ret.mean()
std_ret = monthly_ret.std()

result = pd.DataFrame({
    "cumulative": prices.ffill().iloc[-1] / prices.bfill().iloc[0] - 1,
    "mean_return": mean_ret,
    "std_dev": std_ret,
    "risk_adjusted": (mean_ret / std_ret).where(std_ret > 0),
})

print(result.sort_values("risk_adjusted", ascending=True).head(55))


# ---------------------------------------------------------------------------
# Secondary assessment: among the worst risk-adjusted-momentum names, rank by
# how their trailing P/E has been trending (reconstructed from quarterly
# diluted EPS + historical price), not just its current level.
# ---------------------------------------------------------------------------

TOP_N = 50
MIN_QUARTERS = 2  # minimum number of TTM P/E points required to keep a name

top_neg_momentum = result.sort_values("risk_adjusted", ascending=True).head(TOP_N)
tickers2 = list(top_neg_momentum.index)

pe_cache = Path(f"pe_trend_{start}_{end}.pkl")


def get_pe_trend(ticker, min_quarters=MIN_QUARTERS):
    try:
        tk = yf.Ticker(ticker)
        qf = tk.quarterly_income_stmt
        if qf is None or qf.empty:
            print(f"{ticker}: quarterly_income_stmt empty/None")
            return None
        if "Diluted EPS" not in qf.index:
            print(f"{ticker}: no 'Diluted EPS' row, available: {list(qf.index)}")
            return None

        eps_q = qf.loc["Diluted EPS"].dropna()
        eps_q.index = pd.to_datetime(eps_q.index, errors="coerce")
        eps_q = eps_q[eps_q.index.notna()].sort_index()

        if len(eps_q) < 4:
            print(f"{ticker}: only {len(eps_q)} quarters of EPS, need >=4")
            return None

        ttm_eps = eps_q.rolling(4).sum().dropna()
        if len(ttm_eps) < min_quarters:
            print(f"{ticker}: only {len(ttm_eps)} TTM EPS points, need >={min_quarters}")
            return None

        px_start = (ttm_eps.index.min() - pd.Timedelta(days=10)).strftime("%Y-%m-%d")
        px_end = (ttm_eps.index.max() + pd.Timedelta(days=10)).strftime("%Y-%m-%d")
        px = tk.history(start=px_start, end=px_end, auto_adjust=True)["Close"]
        if px.empty:
            print(f"{ticker}: no price history in {px_start}..{px_end}")
            return None
        px.index = px.index.tz_localize(None)

        rows = []
        for dt, eps in ttm_eps.items():
            dt = pd.Timestamp(dt).tz_localize(None)
            avail = px[px.index <= dt]
            if avail.empty or eps <= 0:
                continue
            price = avail.iloc[-1]
            rows.append((dt, price, eps, price / eps))

        if len(rows) < min_quarters:
            print(f"{ticker}: only {len(rows)} usable rows after price match")
            return None

        return pd.DataFrame(rows, columns=["date", "price", "ttm_eps", "pe"]).set_index("date")
    except Exception as e:
        print(f"{ticker}: EXCEPTION {type(e).__name__}: {e}")
        return None


def pe_slope(df):

    if len(df) < 2:
        return np.nan
    x = np.arange(len(df))
    y = df["pe"].values
    slope, _ = np.polyfit(x, y, 1)
    return slope


if pe_cache.exists():
    pe_data = pd.read_pickle(pe_cache)
else:
    pe_data = {}
    with ThreadPoolExecutor(max_workers=10) as ex:
        futures = {ex.submit(get_pe_trend, t): t for t in tickers2}
        for fut in as_completed(futures):
            t = futures[fut]
            df = fut.result()
            if df is not None:
                pe_data[t] = df
    pd.to_pickle(pe_data, pe_cache)

rows2 = []
for t, df in pe_data.items():
    slope = pe_slope(df)
    pe_now = df["pe"].iloc[-1]
    pe_start = df["pe"].iloc[0]
    pct_change = (pe_now / pe_start - 1) if pe_start else np.nan
    rows2.append({
        "ticker": t,
        "risk_adjusted": top_neg_momentum.loc[t, "risk_adjusted"],
        "forward P/E": pe_now,
        "pe_slope": slope,           # negative = P/E has been falling (cheapening)
        "pe_pct_change": pct_change,
        "n_quarters": len(df),
    })

result2 = pd.DataFrame(rows2).set_index("ticker")

dropped = sorted(set(tickers2) - set(result2.index))
if dropped:
    print(f"\nDropped {len(dropped)} of {TOP_N} names for insufficient EPS/price history: {dropped}")

# "Best" P/E movement here = P/E falling fastest (cheapening on an earnings
# basis while price momentum is already negative). Flip ascending=False if
# you instead want names where P/E is RISING fastest (earnings deteriorating
# faster than price, or a re-rating upward).
result2 = result2.sort_values("pe_slope", ascending=True)

print("\nTop 50 negative-momentum names ranked by P/E trend (most cheapening first):")
print(result2.head(50))

# z-score each metric so they're on comparable scales
z_risk = (result2["risk_adjusted"] - result2["risk_adjusted"].mean()) / result2["risk_adjusted"].std()
z_pe_slope = (result2["pe_slope"] - result2["pe_slope"].mean()) / result2["pe_slope"].std()

result2["global_score"] = z_risk + z_pe_slope

# most negative combined score = worst momentum AND fastest-cheapening P/E
top50_global = result2.sort_values("global_score", ascending=True).head(50)

print(top50_global)


        cumulative  mean_return   std_dev  risk_adjusted
Ticker                                                  
FDXF     -0.166053    -0.058694  0.010513      -5.583052
ROL      -0.347400    -0.100452  0.042635      -2.356114
LHX      -0.166031    -0.044153  0.023740      -1.859845
VICI     -0.106815    -0.027757  0.015142      -1.833110
TXT      -0.160304    -0.042385  0.030240      -1.401610
HONA     -0.284829    -0.150043  0.120462      -1.245560
EME      -0.175794    -0.046625  0.037938      -1.228973
MCD      -0.096266    -0.024825  0.020540      -1.208638
WMT      -0.201685    -0.053833  0.047689      -1.128836
GOOGL    -0.117586    -0.030502  0.027232      -1.120103
TSN      -0.132125    -0.034433  0.031343      -1.098573
MLM      -0.160933    -0.042302  0.039546      -1.069682
CRH      -0.197668    -0.052523  0.051272      -1.024381
PWR      -0.165694    -0.043424  0.046789      -0.928082
GOOG     -0.121297    -0.031346  0.034687      -0.903696
ALGN     -0.092154    -0.023601

In [68]:
import yfinance as yf
ba = yf.Ticker("BA")

info = ba.info
info.get("trailingPE")    # current trailing P/E
info.get("trailingEps")   # current trailing EPS
info.get("forwardPE")     # current forward P/E

q = ba.quarterly_income_stmt
eps = q.loc["Diluted EPS"].sort_index()      # oldest -> newest
ttm_eps = eps.rolling(4).sum().dropna()

In [70]:
import yfinance as yf
ba = yf.Ticker("BA")
print(yf.__version__)
print([m for m in dir(ba) if not m.startswith("_")])

1.7.0
['actions', 'analyst_price_targets', 'balance_sheet', 'balancesheet', 'calendar', 'capital_gains', 'cash_flow', 'cashflow', 'dividends', 'earnings', 'earnings_dates', 'earnings_estimate', 'earnings_history', 'eps_revisions', 'eps_trend', 'fast_info', 'financials', 'funds_data', 'get_actions', 'get_analyst_price_targets', 'get_balance_sheet', 'get_balancesheet', 'get_calendar', 'get_capital_gains', 'get_cash_flow', 'get_cashflow', 'get_dividends', 'get_earnings', 'get_earnings_dates', 'get_earnings_estimate', 'get_earnings_history', 'get_eps_revisions', 'get_eps_trend', 'get_fast_info', 'get_financials', 'get_funds_data', 'get_growth_estimates', 'get_history_metadata', 'get_income_stmt', 'get_incomestmt', 'get_info', 'get_insider_purchases', 'get_insider_roster_holders', 'get_insider_transactions', 'get_institutional_holders', 'get_isin', 'get_major_holders', 'get_mutualfund_holders', 'get_news', 'get_recommendations', 'get_recommendations_summary', 'get_revenue_estimate', 'get_se

In [86]:
import yfinance as yf
ba = yf.Ticker("BA")

val = ba.valuation                  # property
# or
   # method

print(val)
print(type(val))

                               Current     6/30/2026     3/31/2026  \
Market Cap                1.580187e+11  1.709664e+11  1.568191e+11   
Enterprise Value          1.839027e+11  1.972764e+11  1.818581e+11   
Trailing P/E              7.112230e+01  8.556127e+01  8.025403e+01   
Forward P/E               4.878050e+01  8.333333e+02  1.408451e+02   
PEG Ratio (5yr expected)  1.472300e+00  2.345290e+01           NaN   
Price/Sales               1.639480e+00  1.822222e+00  1.695903e+00   
Price/Book                2.564358e+01  2.858492e+01  2.878471e+01   
Enterprise Value/Revenue  1.937900e+00  2.140000e+00  2.032800e+00   
Enterprise Value/EBITDA   2.413940e+01  2.693560e+01  2.471230e+01   

                            12/31/2025     9/30/2025     6/30/2025  
Market Cap                1.703737e+11  1.639604e+11  1.583257e+11  
Enterprise Value          2.007487e+11  1.943224e+11  1.882757e+11  
Trailing P/E                       NaN           NaN           NaN  
Forward P/E            

In [79]:
import pandas as pd

pe = val.loc["Forward P/E"]
current_pe = pe["Current"]          # keep the current value separately
pe = pe.drop("Current")
pe.index = pd.to_datetime(pe.index)
pe = pe.astype(float)

print("Current:", current_pe)
print(pe)

Current: 48.7805
2026-06-30    833.3333
2026-03-31    140.8451
2025-12-31    107.5269
2025-09-30     69.4444
2025-06-30         NaN
Name: Forward P/E, dtype: float64
